# Transformação PNAD COVID com PySpark

Este notebook usa PySpark para transformar os arquivos PNAD COVID, adequado para execução na AWS (EMR/Glue).


In [ ]:
# Configurar Spark Session
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit, expr
from pyspark.sql.types import *
import re

# Criar SparkSession
spark = SparkSession.builder \
    .appName("PNAD_COVID_Transform") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

print("Spark Session criada com sucesso!")
print(f"Spark Version: {spark.version}")


In [ ]:
# Lista de colunas específicas a serem transformadas (do arquivo codigos.ipynb)
colunas_especificas = [
    'A005',
    'B0011', 'B0012', 'B0013', 'B0014', 'B0015', 'B0016', 'B0017', 'B0018', 'B0019',
    'B00110', 'B00111', 'B00112', 'B00113',
    'B002', 'B0031',
    'B0041', 'B0042', 'B0043', 'B0044', 'B0045', 'B0046',
    'B005', 'B006', 'B008',
    'B0101', 'B0102', 'B0103', 'B0104', 'B0105', 'B0106',
    'B011',
    'C001', 'C01012', 'C01022', 'C011A12', 'C011A22', 'C012', 'C013',
    'D0013', 'D0023', 'D0033', 'D0043', 'D0053', 'D0063',
    'E001',
    'F001', 'F0021', 'F002A1', 'F002A2', 'F002A3', 'F002A4', 'F002A5'
]

print(f"Total de colunas específicas: {len(colunas_especificas)}")


In [ ]:
def transformar_arquivo_pnad(spark, caminho_arquivo, colunas_especificas):
    """
    Transforma um arquivo PNAD COVID usando Spark.
    
    Args:
        spark: SparkSession
        caminho_arquivo: caminho para o arquivo CSV
        colunas_especificas: lista de colunas a serem transformadas
    
    Returns:
        DataFrame transformado
    """
    print(f"\n{'='*80}")
    print(f"Processando: {caminho_arquivo}")
    print(f"{'='*80}")
    
    # Carregar o arquivo CSV
    df = spark.read.option("header", "true") \
                  .option("inferSchema", "true") \
                  .csv(caminho_arquivo)
    
    print(f"Shape original: ({df.count()}, {len(df.columns)})")
    print(f"Colunas originais: {df.columns[:10]}...")
    
    # Identificar colunas que existem no DataFrame
    colunas_vdf = [c for c in colunas_especificas if c in df.columns]
    colunas_nao_encontradas = [c for c in colunas_especificas if c not in df.columns]
    
    print(f"\nColunas específicas encontradas: {len(colunas_vdf)}")
    if colunas_nao_encontradas:
        print(f"Colunas não encontradas: {len(colunas_nao_encontradas)}")
        print(f"Exemplos: {colunas_nao_encontradas[:5]}")
    
    # Padrão para identificar colunas V, D, F, A, C, B, E
    padrao_vdf = re.compile(r'^(V|D|F|A|C|B|E)[\dA-Z]')
    
    # Identificar todas as colunas V, D, F, A, C, B, E (excluindo 'Ano' e 'CAPITAL')
    todas_colunas_vdf = [c for c in df.columns 
                         if padrao_vdf.match(c) and c not in ['Ano', 'CAPITAL']]
    
    # Colunas para remover (não estão na lista específica)
    colunas_para_remover = [c for c in todas_colunas_vdf if c not in colunas_vdf]
    
    # Colunas de identificação (não começam com V, D, F, A, C, B, E seguido de números/letras)
    colunas_outras = [c for c in df.columns 
                     if (not padrao_vdf.match(c) or c in ['Ano', 'CAPITAL']) 
                     and c not in colunas_vdf]
    
    print(f"\nColunas V/D/F/A/C/B/E a remover: {len(colunas_para_remover)}")
    print(f"Colunas de identificação: {len(colunas_outras)}")
    print(f"Exemplos de identificação: {colunas_outras[:5]}")
    
    # Remover colunas não desejadas
    colunas_manter = colunas_outras + colunas_vdf
    df_filtrado = df.select(*colunas_manter)
    
    print(f"\nApós remover colunas: ({df_filtrado.count()}, {len(df_filtrado.columns)})")
    
    # 1. Transformar coluna Ano em colunas dummy
    anos_unicos = [row['Ano'] for row in df_filtrado.select('Ano').distinct().collect()]
    print(f"\nAnos únicos encontrados: {anos_unicos}")
    
    # Criar colunas dummy para Ano
    df_com_ano_dummy = df_filtrado
    for ano in anos_unicos:
        df_com_ano_dummy = df_com_ano_dummy.withColumn(
            f'Ano_{int(ano)}',
            when(col('Ano') == ano, 1).otherwise(0)
        )
    
    # Remover coluna Ano original
    df_com_ano_dummy = df_com_ano_dummy.drop('Ano')
    
    print(f"Após criar colunas dummy de Ano: ({df_com_ano_dummy.count()}, {len(df_com_ano_dummy.columns)})")
    
    # 2. Fazer unpivot (melt) das colunas específicas usando stack
    # Selecionar colunas de identificação (excluindo as colunas específicas)
    colunas_id = [c for c in df_com_ano_dummy.columns 
                  if c not in colunas_vdf]
    
    print(f"\nFazendo unpivot de {len(colunas_vdf)} colunas específicas...")
    
    # Filtrar colunas que realmente existem
    colunas_vdf_final = [c for c in colunas_vdf if c in df_com_ano_dummy.columns]
    
    # Criar expressão stack para unpivot
    # Formato: stack(n, 'col1', col1, 'col2', col2, ...) as (Variavel, Valor)
    stack_parts = []
    for c in colunas_vdf_final:
        stack_parts.append(f"'{c}'")
        stack_parts.append(f"`{c}`")
    
    stack_expr = f"stack({len(colunas_vdf_final)}, {', '.join(stack_parts)}) as (Variavel, Valor)"
    
    # Aplicar stack - selecionar colunas de ID e aplicar stack
    df_exploded = df_com_ano_dummy.select(
        [col(c) for c in colunas_id] + [expr(stack_expr)]
    ).select([col(c) for c in colunas_id] + [col('Variavel'), col('Valor')])
    
    print(f"\nShape após transformação: ({df_exploded.count()}, {len(df_exploded.columns)})")
    
    return df_exploded

print("Função de transformação definida!")


In [ ]:
# Lista de arquivos para processar
arquivos_csv = [
    'dados/PNAD_COVID_052020.csv',
    'dados/PNAD_COVID_062020.csv',
    'dados/PNAD_COVID_072020.csv',
    'dados/PNAD_COVID_082020.csv',
    'dados/PNAD_COVID_092020.csv',
    'dados/PNAD_COVID_102020.csv',
    'dados/PNAD_COVID_112020.csv'
]

# Processar cada arquivo
dfs_transformados = {}

for arquivo in arquivos_csv:
    try:
        df_transformado = transformar_arquivo_pnad(spark, arquivo, colunas_especificas)
        
        # Extrair nome do arquivo para salvar
        nome_arquivo = arquivo.split('/')[-1].replace('.csv', '_transformado.csv')
        dfs_transformados[nome_arquivo] = df_transformado
        
        print(f"\n✓ Arquivo {arquivo} processado com sucesso!")
        
    except Exception as e:
        print(f"\n✗ Erro ao processar {arquivo}: {e}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*80}")
print(f"Total de arquivos processados: {len(dfs_transformados)}")
print(f"{'='*80}")


In [ ]:
# Visualizar resultado de um arquivo
if dfs_transformados:
    primeiro_arquivo = list(dfs_transformados.keys())[0]
    df_exemplo = dfs_transformados[primeiro_arquivo]
    
    print(f"\nVisualizando: {primeiro_arquivo}")
    print(f"Schema:")
    df_exemplo.printSchema()
    
    print(f"\nPrimeiras 10 linhas:")
    df_exemplo.show(10, truncate=False)
    
    print(f"\nEstatísticas:")
    print(f"Total de linhas: {df_exemplo.count():,}")
    print(f"Total de colunas: {len(df_exemplo.columns)}")
    print(f"\nColunas: {df_exemplo.columns}")


In [ ]:
# Salvar os DataFrames transformados
# Ajuste o caminho conforme necessário:
# - Para S3: "s3://seu-bucket/pnad-covid-transformado/"
# - Para local: "pnad_transformado/"

caminho_saida = "pnad_transformado/"  # Ajuste para seu bucket S3 se necessário

for nome_arquivo, df in dfs_transformados.items():
    try:
        caminho_completo = f"{caminho_saida}{nome_arquivo}"
        
        print(f"\nSalvando: {caminho_completo}")
        
        # Salvar como Parquet (mais eficiente) ou CSV
        # Opção 1: Parquet (recomendado para Spark)
        df.write.mode('overwrite').parquet(caminho_completo.replace('.csv', '.parquet'))
        
        # Opção 2: CSV
        # df.write.mode('overwrite').option('header', 'true').csv(caminho_completo)
        
        print(f" {nome_arquivo} salvo com sucesso!")
        
    except Exception as e:
        print(f"Erro ao salvar {nome_arquivo}: {e}")

print(f"\n{'='*80}")
print("Processo de salvamento concluído!")
print(f"{'='*80}")


In [ ]:
# Fechar Spark Session
spark.stop()
print("Spark Session encerrada.")
